In [1]:
import pandas as pd
import numpy as np
import networkx as nx

from node2vec import Node2Vec

from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

from xgboost import XGBRegressor

print("Imports successful")

c:\Users\MANIDEEP\AppData\Local\Python\pythoncore-3.11-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Imports successful


In [2]:
df = pd.read_csv(
    "../data/processed/delivery_data_features.csv"
)

print(df.shape)
df.head()

(142502, 41)


,data,trip_creation_time,route_schedule_uuid,route_type,trip_uuid,source_center,source_name,destination_center,destination_name,od_start_time,...,destination_hub_volume,corridor_avg_actual_time,corridor_avg_osrm_time,corridor_avg_distance,historical_corridor_delay,historical_delay_ratio,trip_segments,trip_total_distance,trip_total_actual_time,trip_avg_speed
0,training,2018-09-20 02:35:36.476840,thanos::sroute:eb7bfc78-b351-4c0e-a951-fa3d5c3...,Carting,trip-153741093647649320,IND388121AAA,Anand_VUNagar_DC (Gujarat),IND388620AAB,Khambhat_MotvdDPP_D (Gujarat),2018-09-20 03:21:32.418600,...,54,43.574074,27.685185,32.264602,15.888889,1.573913,10,102.7106,167.0,36.902012
1,training,2018-09-20 02:35:36.476840,thanos::sroute:eb7bfc78-b351-4c0e-a951-fa3d5c3...,Carting,trip-153741093647649320,IND388121AAA,Anand_VUNagar_DC (Gujarat),IND388620AAB,Khambhat_MotvdDPP_D (Gujarat),2018-09-20 03:21:32.418600,...,54,43.574074,27.685185,32.264602,15.888889,1.573913,10,102.7106,167.0,36.902012
2,training,2018-09-20 02:35:36.476840,thanos::sroute:eb7bfc78-b351-4c0e-a951-fa3d5c3...,Carting,trip-153741093647649320,IND388121AAA,Anand_VUNagar_DC (Gujarat),IND388620AAB,Khambhat_MotvdDPP_D (Gujarat),2018-09-20 03:21:32.418600,...,54,43.574074,27.685185,32.264602,15.888889,1.573913,10,102.7106,167.0,36.902012
3,training,2018-09-20 02:35:36.476840,thanos::sroute:eb7bfc78-b351-4c0e-a951-fa3d5c3...,Carting,trip-153741093647649320,IND388121AAA,Anand_VUNagar_DC (Gujarat),IND388620AAB,Khambhat_MotvdDPP_D (Gujarat),2018-09-20 03:21:32.418600,...,54,43.574074,27.685185,32.264602,15.888889,1.573913,10,102.7106,167.0,36.902012
4,training,2018-09-20 02:35:36.476840,thanos::sroute:eb7bfc78-b351-4c0e-a951-fa3d5c3...,Carting,trip-153741093647649320,IND388121AAA,Anand_VUNagar_DC (Gujarat),IND388620AAB,Khambhat_MotvdDPP_D (Gujarat),2018-09-20 03:21:32.418600,...,54,43.574074,27.685185,32.264602,15.888889,1.573913,10,102.7106,167.0,36.902012


In [3]:
G = nx.from_pandas_edgelist(
    df,
    source="source_center",
    target="destination_center",
    create_using=nx.DiGraph()
)

print("Nodes:", G.number_of_nodes())
print("Edges:", G.number_of_edges())

Nodes: 1657
Edges: 2783


In [4]:
node2vec = Node2Vec(
    G,
    dimensions=32,
    walk_length=30,
    num_walks=100,
    workers=4,
    seed=42
)

model = node2vec.fit(
    window=10,
    min_count=1
)

print("Node2Vec training completed")

Computing transition probabilities: 100%|██████████| 1657/1657 [00:00<00:00, 18946.36it/s]


Node2Vec training completed


In [5]:
embeddings = {}

for node in G.nodes():
    embeddings[node] = model.wv[str(node)]

print(
    "Embeddings created:",
    len(embeddings)
)

Embeddings created: 1657


In [6]:
for i in range(32):
    df[f"src_emb_{i}"] = (
        df["source_center"]
        .map(lambda x: embeddings[x][i])
    )

print("Source embeddings added")

Source embeddings added


In [7]:
for i in range(32):
    df[f"dst_emb_{i}"] = (
        df["destination_center"]
        .map(lambda x: embeddings[x][i])
    )

print("Destination embeddings added")

Destination embeddings added


In [8]:
df.to_csv(
    "../data/processed/delivery_data_node2vec.csv",
    index=False
)

print("Enhanced dataset saved")

Enhanced dataset saved


In [9]:
print(df.shape)

embedding_cols = [
    col
    for col in df.columns
    if "emb" in col
]

print(
    "Embedding columns:",
    len(embedding_cols)
)

(142502, 105)
Embedding columns: 64


In [11]:
leakage_cols = [
    "segment_actual_time",
    "trip_total_actual_time",
    "corridor_avg_actual_time"
]

drop_cols = [
    "actual_time",
    "trip_uuid",
    "source_name",
    "destination_name",
    "corridor",
    "data",
    "trip_creation_time",
    "od_start_time",
    "od_end_time",
    "cutoff_timestamp"
] + leakage_cols

target = "actual_time"

print("Ready")

Ready


In [12]:
graph_features = [
    col
    for col in df.columns
    if col not in drop_cols
]

print("Graph Features:", len(graph_features))

Graph Features: 92


In [13]:
baseline_features = [
    col
    for col in graph_features
    if not col.startswith("src_emb_")
    and not col.startswith("dst_emb_")
]

print("Baseline Features:", len(baseline_features))

Baseline Features: 28


In [14]:
X_graph = df[graph_features]
X_base = df[baseline_features]

y = df[target]

print(X_graph.shape)
print(X_base.shape)

(142502, 92)
(142502, 28)


In [15]:
groups = df["trip_uuid"]

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(X_graph, y, groups=groups)
)

X_graph_train = X_graph.iloc[train_idx]
X_graph_test = X_graph.iloc[test_idx]

X_base_train = X_base.iloc[train_idx]
X_base_test = X_base.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

print(X_graph_train.shape)
print(X_base_train.shape)

(114541, 92)
(114541, 28)


In [18]:
print(
    X_base_train.select_dtypes(
        include=["object", "string"]
    ).columns.tolist()
)

['route_schedule_uuid', 'route_type', 'source_center', 'destination_center']


In [19]:
print(
    X_base_train.select_dtypes(
        include=["object", "string"]
    ).columns.tolist()
)

['route_schedule_uuid', 'route_type', 'source_center', 'destination_center']


In [20]:
X_base_train.dtypes[
    X_base_train.dtypes.astype(str).str.contains("str|object")
]

route_schedule_uuid    str
route_type             str
source_center          str
destination_center     str
dtype: object

In [21]:
X_base_train[
    [
        "route_schedule_uuid",
        "route_type",
        "source_center",
        "destination_center"
    ]
].head()

,route_schedule_uuid,route_type,source_center,destination_center
0,thanos::sroute:eb7bfc78-b351-4c0e-a951-fa3d5c3...,Carting,IND388121AAA,IND388620AAB
1,thanos::sroute:eb7bfc78-b351-4c0e-a951-fa3d5c3...,Carting,IND388121AAA,IND388620AAB
2,thanos::sroute:eb7bfc78-b351-4c0e-a951-fa3d5c3...,Carting,IND388121AAA,IND388620AAB
3,thanos::sroute:eb7bfc78-b351-4c0e-a951-fa3d5c3...,Carting,IND388121AAA,IND388620AAB
4,thanos::sroute:eb7bfc78-b351-4c0e-a951-fa3d5c3...,Carting,IND388121AAA,IND388620AAB


In [22]:
for col in [
    "route_schedule_uuid",
    "route_type",
    "source_center",
    "destination_center"
]:
    df[col] = pd.factorize(df[col])[0]

In [23]:
print(
    df[
        [
            "route_schedule_uuid",
            "route_type",
            "source_center",
            "destination_center"
        ]
    ].dtypes
)

route_schedule_uuid    int64
route_type             int64
source_center          int64
destination_center     int64
dtype: object


In [24]:
graph_features = [
    col
    for col in df.columns
    if col not in drop_cols
]

print("Graph Features:", len(graph_features))

Graph Features: 92


In [25]:
baseline_features = [
    col
    for col in graph_features
    if not col.startswith("src_emb_")
    and not col.startswith("dst_emb_")
]

print("Baseline Features:", len(baseline_features))

Baseline Features: 28


In [26]:
X_graph = df[graph_features]
X_base = df[baseline_features]

y = df[target]

print(X_graph.shape)
print(X_base.shape)

(142502, 92)
(142502, 28)


In [27]:
groups = df["trip_uuid"]

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(X_graph, y, groups=groups)
)

X_graph_train = X_graph.iloc[train_idx]
X_graph_test = X_graph.iloc[test_idx]

X_base_train = X_base.iloc[train_idx]
X_base_test = X_base.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

print(X_graph_train.shape)
print(X_base_train.shape)

(114541, 92)
(114541, 28)


In [28]:
baseline_model = XGBRegressor(
    n_estimators=300,
    max_depth=8,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

baseline_model.fit(
    X_base_train,
    y_train
)

print("Baseline trained")

Baseline trained


In [29]:
graph_model = XGBRegressor(
    n_estimators=300,
    max_depth=8,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

graph_model.fit(
    X_graph_train,
    y_train
)

print("Graph model trained")

Graph model trained


In [30]:
baseline_preds = baseline_model.predict(
    X_base_test
)

print("Baseline predictions done")

Baseline predictions done


In [31]:
graph_preds = graph_model.predict(
    X_graph_test
)

print("Graph predictions done")

Graph predictions done


In [32]:
baseline_mae = mean_absolute_error(
    y_test,
    baseline_preds
)

baseline_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        baseline_preds
    )
)

baseline_r2 = r2_score(
    y_test,
    baseline_preds
)

baseline_within15 = (
    np.abs(
        baseline_preds - y_test
    )
    <= (0.15 * y_test)
).mean() * 100

print("Baseline MAE:", round(baseline_mae,2))
print("Baseline RMSE:", round(baseline_rmse,2))
print("Baseline R2:", round(baseline_r2,4))
print("Baseline Within15:", round(baseline_within15,2))

Baseline MAE: 5.4
Baseline RMSE: 21.86
Baseline R2: 0.9987
Baseline Within15: 99.54


In [33]:
graph_mae = mean_absolute_error(
    y_test,
    graph_preds
)

graph_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        graph_preds
    )
)

graph_r2 = r2_score(
    y_test,
    graph_preds
)

graph_within15 = (
    np.abs(
        graph_preds - y_test
    )
    <= (0.15 * y_test)
).mean() * 100

print("Graph MAE:", round(graph_mae,2))
print("Graph RMSE:", round(graph_rmse,2))
print("Graph R2:", round(graph_r2,4))
print("Graph Within15:", round(graph_within15,2))

Graph MAE: 5.18
Graph RMSE: 21.27
Graph R2: 0.9987
Graph Within15: 99.61


In [34]:
comparison = pd.DataFrame({
    "Metric":[
        "MAE",
        "RMSE",
        "R2",
        "Within15%"
    ],
    "Baseline":[
        baseline_mae,
        baseline_rmse,
        baseline_r2,
        baseline_within15
    ],
    "Node2Vec":[
        graph_mae,
        graph_rmse,
        graph_r2,
        graph_within15
    ]
})

comparison

,Metric,Baseline,Node2Vec
0,MAE,5.399303,5.177449
1,RMSE,21.858203,21.271721
2,R2,0.998669,0.998739
3,Within15%,99.535067,99.610171


In [35]:
sorted(
    [
        c for c in X_graph.columns
        if "delay" in c.lower()
        or "factor" in c.lower()
        or "actual" in c.lower()
    ]
)

['actual_distance_to_destination',
 'cutoff_factor',
 'factor',
 'historical_corridor_delay',
 'historical_delay_ratio',
 'segment_factor']